# Ambara — Step-by-Step Colab Runner

Each pipeline step has its own cell. Run **Environment Setup** once, then execute
any step independently — no need to run everything in sequence.

**Workflow overview:**
1. Fill in the config cell and run **Environment Setup**
2. Pick any step cell, set its variables, and run it

| Step | What it does |
|------|--------------|
| 1 · Download | Pull audio from a YouTube URL |
| 2 · Extract | VAD → classify → transcribe → write clips |
| 3 · VAD Only | Detect speech segments and export as JSON |
| 4 · Sync | Upload a local extraction run to Supabase |
| 5 · Export Training Data | Pull corrected clips from Supabase as a training dataset |
| 6 · Train | Fine-tune Whisper on the exported dataset |
| 7 · Re-draft | Re-transcribe pending clips with the fine-tuned model |

- Supabase credentials come from `.env` stored in Google Drive (mounted once).
- Trained models can be pushed to HuggingFace Hub automatically.

In [ ]:
# ---- Google Drive (only for .env) ----
DRIVE_ROOT = "ambara"

# ---- Repository ----
REPO_URL = "https://github.com/ny-randriantsarafara/ny-feoko.git"
REPO_BRANCH = "main"

# ---- HuggingFace ----
HF_TOKEN = ""  # Write token — required for --push-to-hub

## Environment Setup

Run this cell once per session. Mounts Drive (for `.env`), clones/updates the repo,
and installs dependencies.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
DRIVE_BASE = DRIVE_MOUNT / "MyDrive" / DRIVE_ROOT
REPO_DIR = Path("/content/ny-feoko")

# ---- Mount Google Drive (for .env) ----
if not (DRIVE_MOUNT / "MyDrive").exists():
    drive.mount(str(DRIVE_MOUNT))
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"Drive mounted at {DRIVE_MOUNT}")

# ---- Clone or update repo ----
if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
    print(f"Repo updated: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print(f"Repo cloned: {REPO_DIR}")

os.chdir(REPO_DIR)

# ---- Symlink .env from Drive (if present) ----
env_drive = DRIVE_BASE / ".env"
env_local = REPO_DIR / ".env"
if env_drive.exists():
    if env_local.is_symlink() or env_local.exists():
        env_local.unlink()
    env_local.symlink_to(env_drive)
    print(f"  .env -> {env_drive}")
else:
    print("WARNING: No .env found on Drive. Place it at My Drive/ambara/.env")

# ---- Python version check ----
v = sys.version_info
print(f"\nPython {v.major}.{v.minor}.{v.micro}")
if v < (3, 11):
    print("WARNING: This project requires Python >= 3.11.")

# ---- Install dependencies ----
subprocess.run(["make", "colab-install"], check=True, cwd=str(REPO_DIR))

# ---- HuggingFace login ----
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace Hub.")

# ---- Environment summary ----
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
print(f"\n{'=' * 50}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()} ({gpu_name})")
print(f"Repo:    {REPO_DIR}")
print(f"{'=' * 50}")
print("Setup complete.")

---
## Steps

Each cell below is independent. Set its variables and run it whenever you need that step.

## 1 · Download

Pull audio from a YouTube URL and save it as a 16 kHz mono WAV.

In [ ]:
import subprocess

DL_URL    = "https://youtube.com/watch?v=..."  # YouTube URL
DL_LABEL  = ""                                  # Output filename (defaults to video title)
DL_OUTPUT = "data/input"                        # Output directory
DRY_RUN   = False                               # Set True to validate without downloading

cmd = ["python", "-m", "yt_download.cli", "download", DL_URL, "-o", DL_OUTPUT]
if DL_LABEL:
    cmd += ["-l", DL_LABEL]
if DRY_RUN:
    cmd += ["--dry-run"]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

## 2 · Extract

Run the full extraction pipeline on a local audio file: VAD → classify → transcribe → write clips.

In [ ]:
import subprocess

EX_INPUT            = "data/input/audio.wav"  # Input audio file
EX_OUTPUT           = "data/output"            # Output directory
EX_LABEL            = ""                       # Run label for output subdirectory
EX_DEVICE           = "cuda"                   # cuda | mps | cpu
EX_WHISPER_MODEL    = "small"                  # Whisper size (tiny/base/small/medium/large)
EX_WHISPER_HF       = ""                       # HuggingFace model ID (overrides EX_WHISPER_MODEL)
EX_VAD_THRESHOLD    = 0.35                     # VAD speech probability threshold
EX_SPEECH_THRESHOLD = 0.35                     # Minimum speech classifier score
DRY_RUN             = False                    # Set True to load models and process, but skip writing files

cmd = [
    "python", "-m", "clip_extraction.cli", "run",
    "-i", EX_INPUT,
    "-o", EX_OUTPUT,
    "--device", EX_DEVICE,
    "--whisper-model", EX_WHISPER_MODEL,
    "--vad-threshold", str(EX_VAD_THRESHOLD),
    "--speech-threshold", str(EX_SPEECH_THRESHOLD),
    "--verbose",
]
if EX_LABEL:
    cmd += ["-l", EX_LABEL]
if EX_WHISPER_HF:
    cmd += ["--whisper-hf", EX_WHISPER_HF]
if DRY_RUN:
    cmd += ["--dry-run"]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

## 3 · VAD Only

Detect speech segments without transcribing. Outputs a JSON file of timestamps.

In [ ]:
import subprocess

VAD_INPUT     = "data/input/audio.wav"        # Input audio file
VAD_OUTPUT    = "data/vad_segments.json"       # Output JSON file
VAD_THRESHOLD = 0.35                           # Speech probability threshold
DRY_RUN       = False                          # Set True to detect segments but skip writing JSON

cmd = [
    "python", "-m", "clip_extraction.cli", "vad-only",
    "-i", VAD_INPUT,
    "-o", VAD_OUTPUT,
    "--vad-threshold", str(VAD_THRESHOLD),
]
if DRY_RUN:
    cmd += ["--dry-run"]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

## 4 · Sync

Upload a local extraction run directory (clips + metadata) to Supabase.

In [ ]:
import subprocess

SYNC_DIR   = "data/output/my-run"  # Path to extraction run directory
SYNC_LABEL = ""                    # Run label (defaults to directory name)
DRY_RUN    = False                 # Set True to validate files but skip uploading to Supabase

cmd = ["python", "-m", "db_sync.cli", "sync", "-d", SYNC_DIR]
if SYNC_LABEL:
    cmd += ["-l", SYNC_LABEL]
if DRY_RUN:
    cmd += ["--dry-run"]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

## 5 · Export Training Data

Pull corrected clips from Supabase and build a HuggingFace-compatible training dataset.

In [ ]:
import subprocess

EXPORT_LABEL      = ""               # Run label in Supabase
EXPORT_SOURCE_DIR = "data/output"    # Local directory containing the clips/ subdirectory
EXPORT_OUTPUT     = "data/training"  # Parent directory for the exported dataset
EXPORT_EVAL_SPLIT = 0.1              # Fraction reserved for evaluation (0.0 – 0.5)
DRY_RUN           = False            # Set True to query clips but skip writing files

if not EXPORT_LABEL:
    raise ValueError("Set EXPORT_LABEL to the run label from Supabase.")

cmd = [
    "python", "-m", "db_sync.cli", "export-training",
    "-l", EXPORT_LABEL,
    "-d", EXPORT_SOURCE_DIR,
    "-o", EXPORT_OUTPUT,
    "--eval-split", str(EXPORT_EVAL_SPLIT),
]
if DRY_RUN:
    cmd += ["--dry-run"]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

## 6 · Train

Fine-tune Whisper on the exported training dataset.

In [ ]:
import subprocess

TRAIN_DATA_DIR    = "data/training"          # Path to exported training dataset
TRAIN_BASE_MODEL  = "openai/whisper-small"   # HuggingFace model to fine-tune
TRAIN_DEVICE      = "cuda"                   # cuda | mps | cpu | auto
TRAIN_EPOCHS      = 10
TRAIN_BATCH_SIZE  = 4
TRAIN_LR          = 1e-5
TRAIN_OUTPUT_DIR  = "models/whisper-mg-v1"   # Local save path
TRAIN_PUSH_TO_HUB = ""                       # HuggingFace repo ID (e.g. user/whisper-mg)
DRY_RUN           = False                    # Set True to load model and dataset but skip training

cmd = [
    "python", "-m", "asr_training.cli", "train",
    "-d", TRAIN_DATA_DIR,
    "-o", TRAIN_OUTPUT_DIR,
    "--device", TRAIN_DEVICE,
    "--base-model", TRAIN_BASE_MODEL,
    "--epochs", str(TRAIN_EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--lr", str(TRAIN_LR),
]
if TRAIN_PUSH_TO_HUB:
    cmd += ["--push-to-hub", TRAIN_PUSH_TO_HUB]
if DRY_RUN:
    cmd += ["--dry-run"]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

## 7 · Re-draft

Re-transcribe pending clips with the fine-tuned model and update Supabase drafts.

In [ ]:
import subprocess

REDRAFT_MODEL      = "models/whisper-mg-v1"  # Local model path or HuggingFace repo ID
REDRAFT_SOURCE_DIR = "data/output"           # Directory containing the clips/ subdirectory
REDRAFT_LABEL      = ""                      # Run label in Supabase
REDRAFT_DEVICE     = "cuda"                  # cuda | mps | cpu | auto
DRY_RUN            = False                   # Set True to load model but skip transcription and DB updates

if not REDRAFT_LABEL:
    raise ValueError("Set REDRAFT_LABEL to the run label from Supabase.")

cmd = [
    "python", "-m", "asr_training.cli", "re-draft",
    "-m", REDRAFT_MODEL,
    "-d", REDRAFT_SOURCE_DIR,
    "-l", REDRAFT_LABEL,
    "--device", REDRAFT_DEVICE,
]
if DRY_RUN:
    cmd += ["--dry-run"]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)